In [27]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GlobalAveragePooling1D,
    Dense
)

# Load the Preprocessed Data

In [28]:
X_train_padded = np.load(
    "../Dataset/processed/X_train_padded.npy"
)

X_val_padded = np.load(
    "../Dataset/processed/X_val_padded.npy"
)

X_test_padded = np.load(
    "../Dataset/processed/X_test_padded.npy"
)

y_train = np.load(
    "../Dataset/processed/y_train.npy"
)

y_val = np.load(
    "../Dataset/processed/y_val.npy"
)

y_test = np.load(
    "../Dataset/processed/y_test.npy"
)

In [29]:
print("Training:", X_train_padded.shape)
print("Validation:", X_val_padded.shape)
print("Testing:", X_test_padded.shape)

Training: (34705, 200)
Validation: (7439, 200)
Testing: (7438, 200)


# Load the Tokenizer

In [30]:
with open(
    "../Dataset/processed/tokenizer.pkl",
    "rb"
) as file:

    tokenizer = pickle.load(file)

In [31]:
with open(
    "../Dataset/processed/preprocessing_config.pkl",
    "rb"
) as file:

    preprocessing_config = pickle.load(file)

In [32]:
MAX_SEQUENCE_LENGTH = preprocessing_config[
    "max_sequence_length"
]

NUM_WORDS = preprocessing_config[
    "num_words"
]

print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Vocabulary limit:", NUM_WORDS)

Maximum sequence length: 200
Vocabulary limit: 20000


In [33]:
# Integer Encoding

word_index = tokenizer.word_index

print("Vocabulary size:", len(word_index))

print("\nFirst 20 words:")
print(list(word_index.items())[:20])

Vocabulary size: 85869

First 20 words:
[('<OOV>', 1), ('movie', 2), ('film', 3), ('one', 4), ('like', 5), ('good', 6), ('time', 7), ('even', 8), ('would', 9), ('story', 10), ('really', 11), ('see', 12), ('well', 13), ('much', 14), ('bad', 15), ('get', 16), ('great', 17), ('people', 18), ('also', 19), ('first', 20)]


In [34]:
with open(
    "../data/processed/text_splits.pkl",
    "rb"
) as file:

    text_splits = pickle.load(file)

In [35]:
X_train = text_splits["X_train"]
X_val = text_splits["X_val"]
X_test = text_splits["X_test"]

In [36]:
train_text = X_train
val_text = X_val
test_text = X_test

In [37]:
# TF-IDF Vectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

In [38]:
# Fit TF-IDF Only on Training Data

X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_text
)


X_val_tfidf = tfidf_vectorizer.transform(
    val_text
)

X_test_tfidf = tfidf_vectorizer.transform(
    test_text
)

In [39]:
# check TF-ID shapes

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_val_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)

TF-IDF training shape: (34705, 20000)
TF-IDF validation shape: (7439, 20000)
TF-IDF testing shape: (7438, 20000)


In [40]:
# Inspect TF-ID Features

feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))

print("\nFirst 20 features:")
print(feature_names[:20])

sample_vector = X_train_tfidf[0]

print(sample_vector)

Number of features: 20000

First 20 features:
['aaron' 'abandon' 'abandoned' 'abbey' 'abbot' 'abbott' 'abbott costello'
 'abby' 'abc' 'abducted' 'abilities' 'ability' 'able' 'able find'
 'able get' 'able keep' 'able make' 'able see' 'able watch' 'ably']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 102 stored elements and shape (1, 20000)>
  Coords	Values
  (0, 11723)	0.10435358801726821
  (0, 15339)	0.042211521088489704
  (0, 12454)	0.047791723165993165
  (0, 5500)	0.06426934860743182
  (0, 13855)	0.05080331711433631
  (0, 19684)	0.052598263125894676
  (0, 4818)	0.05988633403719985
  (0, 19988)	0.08376093427385468
  (0, 11668)	0.03880576975370319
  (0, 10729)	0.0495066225698543
  (0, 7017)	0.035488243524137694
  (0, 19153)	0.03867737280127706
  (0, 763)	0.045159743228414734
  (0, 14872)	0.11676008812821088
  (0, 19992)	0.2650788418709953
  (0, 11207)	0.09561833676141122
  (0, 16759)	0.1147847475941255
  (0, 9038)	0.057968086032315035
  (0, 2424)	0.42872027778843014
  (

In [41]:
with open(
    "../data/processed/tfidf_vectorizer.pkl",
    "wb"
) as file:

    pickle.dump(tfidf_vectorizer, file)

In [42]:
# TF-IDF Representation Summary

# Word Embeddinds

vocab_size = min(
    NUM_WORDS,
    len(tokenizer.word_index) + 1
)

print("Vocabulary size used by model:", vocab_size)

Vocabulary size used by model: 20000


In [43]:
# Create a Trainable Embedding Layer

EMBEDDING_DIM = 128

embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    input_length=MAX_SEQUENCE_LENGTH
)

/home/aximsoft/snap/code/258/.local/share/virtualenvs/Text_Sentiment_Analysis-D79r_KZq/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [44]:
# Build a Small Embedding Demonstration Model

embedding_demo_model = Sequential([
    embedding_layer,
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

embedding_demo_model = Sequential([
    embedding_layer,
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [45]:
embedding_demo_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [46]:
# Save Feature-Representation Information

feature_config = {
    "vocab_size": vocab_size,
    "embedding_dim": EMBEDDING_DIM,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "tfidf_max_features": 20000,
    "tfidf_ngram_range": (1, 2)
}

In [47]:
with open(
    "../Dataset/processed/feature_config.pkl",
    "wb"
) as file:

    pickle.dump(feature_config, file)